In [0]:
import subprocess
import requests 
import tempfile
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")
brev_ip = dbutils.secrets.get(scope="brev", key="brev_ip")
pat = dbutils.secrets.get(scope="databricks", key="pat") 
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)


with open("/tmp/ssh_private_key", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key", 0o600)

# step 1 - is instance reachable? 
def check_instance():
    try: 
        result = subprocess.run(
            ["ssh", "-i", "/tmp/ssh_private_key",
             "-o", "StrictHostKeyChecking=no",
             "-o", "ConnectTimeout=10",
             f"ubuntu@{brev_ip}",
             "echo ALIVE"],
            capture_output=True, text=True, timeout=15
        )
        instance_up = result.returncode == 0
        return instance_up
    
    except subprocess.TimeoutExpired:
        return False, False, "TIMEOUT"
    
    except Exception as e:
        return False, False, str(e)

current_job_id = 129636104529500
print(f"Current job ID: {current_job_id}")

# step 2 - any job currently running? 
def check_jobs_running():
    response = requests.get(
        f"{host}/api/2.1/jobs/runs/list",
        headers={"Authorization": f"Bearer {pat}"},
        params={"active_only": True, "limit": 25}
    )
    if response.status_code != 200:
        raise Exception(f"API error: {response.text}")
    
    runs = response.json().get("runs", [])

    active_runs = [
        r for r in runs
        if r.get("state", {}).get("life_cycle_state") == "RUNNING"
        and r.get("job_id") != current_job_id
    ]

    return active_runs  # empty list = no jobs running

def shutdown_instance():
    result = subprocess.run(
        ["ssh", "-i", "/tmp/ssh_private_key",
         "-o", "StrictHostKeyChecking=no",
         "-o", "ConnectTimeout=10",
         f"ubuntu@{brev_ip}",
         "sudo shutdown -h now"],
        capture_output=True, text=True, timeout=15
    )
    return result.returncode == 0

# ── Main logic ───────────────────────────────────────
instance_up = check_instance()
print(f"Instance up: {instance_up}")

if not instance_up:
    print("Instance is OFF — nothing to do")
else:
    print("Instance is ON — checking Databricks jobs...")
    active_runs = check_jobs_running()

    if active_runs:
        print(f"Found {len(active_runs)} active job(s) — instance is busy:")
        for r in active_runs:
            print(f"  - {r.get('run_name', 'unknown')} | job_id={r['job_id']} | state={r['state']['life_cycle_state']}")
    else:
        print("Instance is ON but no jobs running — closing the instance...")
        shutdown_instance()
        